In [ ]:
import json
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
import time

DATA_PATH = "../data/processed/pubmedqa_filtered.json"
with open(DATA_PATH, "r") as f:
    corpus = json.load(f)

print(f"Corpus size: {len(corpus)} samples")
print(f"Keys per sample: {list(corpus[0].keys())}")
print(f"\nSample query: {corpus[0]['query']}")
print(f"Sample abstract preview: {corpus[0]['supporting_abstracts'][0][:200]}")

In [ ]:
documents = []
doc_metadata = []

for sample in corpus:
    for abstract in sample["supporting_abstracts"]:
        documents.append(abstract)
        doc_metadata.append({
            "pubid": sample["pubid"],
            "query": sample["query"],
            "gold_answer": sample["gold_answer"],
            "label": sample["label"]
        })

print(f"Total documents indexed: {len(documents)}")
print(f"Avg abstracts per sample: {len(documents)/len(corpus):.2f}")

print("\nBuilding BM25 index...")
start = time.time()
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)
print(f"BM25 index built in {time.time()-start:.2f}s")
print(f"Corpus vocabulary size: {len(bm25.idf)}")

In [ ]:
def bm25_retrieve(query, k=3):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_k_indices = np.argsort(scores)[::-1][:k]
    
    results = []
    for idx in top_k_indices:
        results.append({
            "abstract": documents[idx],
            "score": scores[idx],
            "pubid": doc_metadata[idx]["pubid"],
            "gold_answer": doc_metadata[idx]["gold_answer"],
            "label": doc_metadata[idx]["label"]
        })
    return results

# Test on 3 real queries from corpus
test_queries = [corpus[0]["query"], corpus[10]["query"], corpus[50]["query"]]

print("=== BM25 RETRIEVAL VALIDATION ===\n")
for query in test_queries:
    print(f"Query: {query}")
    results = bm25_retrieve(query, k=3)
    for i, r in enumerate(results):
        print(f"  [{i+1}] score={r['score']:.3f} | pubid={r['pubid']} | "
              f"abstract preview: {r['abstract'][:100]}")
    print()

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading biomedical sentence encoder...")
enc_model = SentenceTransformer("NeuML/pubmedbert-base-embeddings")
print("Encoder loaded.")

print(f"\nEmbedding {len(documents)} documents...")
start = time.time()
doc_embeddings = enc_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"\nDone in {time.time()-start:.1f}s")
print(f"Embedding matrix shape: {doc_embeddings.shape}")

# Normalize and build FAISS index
faiss.normalize_L2(doc_embeddings)
dim = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings)
print(f"FAISS index built. Total vectors: {index.ntotal}")

In [ ]:
def faiss_retrieve(query, k=3):
    query_embedding = enc_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "abstract": documents[idx],
            "score": float(score),
            "pubid": doc_metadata[idx]["pubid"],
            "gold_answer": doc_metadata[idx]["gold_answer"],
            "label": doc_metadata[idx]["label"]
        })
    return results

# Validate on the same 3 queries, compare BM25 vs FAISS
test_queries = [corpus[0]["query"], corpus[10]["query"], corpus[50]["query"]]

print("=== BM25 vs FAISS RETRIEVAL COMPARISON ===\n")
for query in test_queries:
    print(f"Query: {query}")
    
    bm25_results = bm25_retrieve(query, k=3)
    faiss_results = faiss_retrieve(query, k=3)
    
    print(f"  BM25  [1] score={bm25_results[0]['score']:.3f} | pubid={bm25_results[0]['pubid']} | {bm25_results[0]['abstract'][:80]}")
    print(f"  FAISS [1] score={faiss_results[0]['score']:.3f} | pubid={faiss_results[0]['pubid']} | {faiss_results[0]['abstract'][:80]}")
    print()

In [13]:
def hybrid_retrieve(query, k=3, rrf_k=60):
    bm25_results = bm25_retrieve(query, k=k*2)
    faiss_results = faiss_retrieve(query, k=k*2)
    combined = {}
    
    for rank, r in enumerate(bm25_results):
        key = r["abstract"]
        combined[key] = {"meta": r, "score": 1 / (rrf_k + rank + 1)}
    
    for rank, r in enumerate(faiss_results):
        key = r["abstract"]
        if key in combined:
            combined[key]["score"] += 1 / (rrf_k + rank + 1)
        else:
            combined[key] = {"meta": r, "score": 1 / (rrf_k + rank + 1)}
    
    sorted_results = sorted(combined.values(), key=lambda x: x["score"], reverse=True)[:k]
    return [r["meta"] for r in sorted_results]

def build_rag_prompt(query, k=3, tokenizer=None):
    results = hybrid_retrieve(query, k=k)
    
    context_parts = [f"[{i+1}] {r['abstract']}" for i, r in enumerate(results)]
    context = "\n\n".join(context_parts)
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    span_index = None
    if tokenizer is not None:
        span_index = []
        tokens_so_far = tokenizer("Context:\n", return_tensors="pt")["input_ids"].shape[1]
        
        for i, part in enumerate(context_parts):
            part_len = tokenizer(part, return_tensors="pt")["input_ids"].shape[1]
            sep_len = tokenizer("\n\n", return_tensors="pt")["input_ids"].shape[1]
            span_index.append({
                "doc_rank": i,
                "pubid": results[i]["pubid"],
                "token_start": tokens_so_far,
                "token_end": tokens_so_far + part_len
            })
            tokens_so_far += part_len + sep_len
    
    return prompt, results, span_index

def build_suppressed_prompt(query):
    prompt = f"Question: {query}\nAnswer:"
    return prompt

print("=== RAG PROMPT ASSEMBLY VALIDATION ===\n")
for sample in [corpus[0], corpus[10], corpus[50]]:
    query = sample["query"]
    rag_prompt, retrieved, spans = build_rag_prompt(query, k=3)
    suppressed_prompt = build_suppressed_prompt(query)
    
    print(f"Query: {query}")
    print(f"Retrieved pubids: {[r['pubid'] for r in retrieved]}")
    print(f"RAG prompt length: {len(rag_prompt.split())} words")
    print(f"Suppressed prompt length: {len(suppressed_prompt.split())} words")
    print(f"RAG prompt preview: {rag_prompt[:200]}")
    print()

print("01_rag_pipeline: COMPLETE ✓")

=== RAG PROMPT ASSEMBLY VALIDATION ===

Query: Landolt C and snellen e acuity: differences in strabismus amblyopia?
Retrieved pubids: [16418930, 16418930, 16418930]
RAG prompt length: 261 words
Suppressed prompt length: 12 words
RAG prompt preview: Context:
[1] Differences between Landolt C acuity (LR) and Snellen E acuity (SE) were small. The mean decimal values for LR and SE were 0.25 and 0.29 in the entire group and 0.14 and 0.16 for the eyes

Query: Is there a model to teach and practice retroperitoneoscopic nephrectomy?
Retrieved pubids: [22694248, 22266735, 12153648]
RAG prompt length: 219 words
Suppressed prompt length: 12 words
RAG prompt preview: Context:
[1] Although the retroperitoneal approach has been the preferred choice for open urological procedures, retroperitoneoscopy is not the preferred approach for laparoscopy. This study aims to d

Query: Does the clinical presentation of a prior preterm birth predict risk in a subsequent pregnancy?
Retrieved pubids: [26215326, 26